# Introduction

In this notebook, we preprocess all the files 

In [1]:
import os
import shutil
from ultralytics import YOLO
from nazi_symbols_classification.training.data_preparation import remap_images, get_image_paths
from nazi_symbols_classification.image_processing import (
    auto_resize, grayscale, auto_adjust_contrast,
    randomly_flip_image, randomly_rotate_image, randomly_shear_image,
    randomly_change_image_hue_saturation_brightness,
    randomly_change_image_contrast_brightness, randomly_blur_image,
    randomly_add_noise
)
from nazi_symbols_classification.constants import FlipDirection
from nazi_symbols_classification.pipeline import Pipeline


dir_name = os.path.dirname(os.getcwd())

In [28]:
preprocessing_pipeline = Pipeline([
    ("auto_resize", auto_resize, dict(new_width=640, new_height=640)),
    ("grayscale", grayscale, None),
    ("auto_adjust_contrast", auto_adjust_contrast, None),
])

In [31]:
augmentation_pipeline = Pipeline([
    ("randomly_flip_image", randomly_flip_image,
     dict(directions=(FlipDirection.HORIZONTALLY, FlipDirection.VERTICALLY))),
    ("randomly_rotate_image", randomly_rotate_image, dict(angle_range=15)),
    ("randomly_shear_image", randomly_shear_image,
     dict(vertical_angle_range=10, horizontal_angle_range=10)),
    ("randomly_change_image_hue", randomly_change_image_hue_saturation_brightness,
     dict(hue_range=15, saturation_range=0, brightness_range=0)),
    ("randomly_change_image_saturation",
     randomly_change_image_hue_saturation_brightness,
     dict(hue_range=0, saturation_range=0.25, brightness_range=0)),
    ("randomly_change_image_brightness",
     randomly_change_image_hue_saturation_brightness,
     dict(hue_range=0, saturation_range=0, brightness_range=0.15)),
    ("randomly_change_image_exposure",
     randomly_change_image_contrast_brightness,
     dict(sign=0, contrast_control=1, brightness_range_percentage=0.1)),
    ("randomly_blur_image", randomly_blur_image, dict(ksize_range=7)),
    ("randomly_add_noise", randomly_add_noise, dict(prob_range=0.001)),
])

## Preprocess binary classification datasets
1. run preprocessing pipeline to all images, 
2. then apply augmentation pipeline just for the nazi images in training dataset

In [ ]:
binary_images = get_image_paths(path=f"{dir_name}/datasets/nazi-symbols-detection", sub_folders=["train", "test", "val"])

In [ ]:
preprocessing_pipeline.run(binary_images)

In [ ]:
training_binary_images = [path for path in binary_images if path.startswith(f"{dir_name}/datasets/nazi-symbols-detection/train/nazi-symbol/")]

augmentation_pipeline.run(training_binary_images, skip_prob=0.5)

## Preprocess multi-class classification datasets
1. remap the images
2. run preprocessing pipeline to all images, 
3. then apply augmentation pipeline just for the images in training dataset

In [14]:
regroup = {
    "neo_nazi": ['national_rebirth_poland', 'combat_18_emblem', 'atomwaffen', 
                 'kolovrat', 'volksfront_emblem', 'celtic_cross', 'hammerskins', 
                 'identitaere_bewegung_emblem', 'blood_honor_emblem', 'golden_dawn'],
    "siegrune": ['doppelsiegrune', 'siegrune',]
}

In [21]:
remap_images(regroup, f"{dir_name}/datasets/nazi-symbols-classification", sub_folders = ["train", "test", "val"])

In [23]:
multi_cls_images = get_image_paths(path=f"{dir_name}/datasets/nazi-symbols-classification", sub_folders=["train", "test", "valid"])
len(multi_cls_images)

2199

In [29]:
preprocessing_pipeline.run(multi_cls_images)

In [30]:
training_multi_cls_images = [path for path in multi_cls_images if path.startswith(f"{dir_name}/datasets/nazi-symbols-classification/train/")]
len(training_multi_cls_images)

1411

In [32]:
augmentation_pipeline.run(training_multi_cls_images, skip_prob=0.5)

In [33]:
multi_cls_images = get_image_paths(path=f"{dir_name}/datasets/nazi-symbols-classification", sub_folders=["train", "test", "val"])
len(multi_cls_images)

8683